# SSB StatBank → Standardize Layer

Leser JSON-filer fra Landing layer og konverterer til flate Delta-tabeller i Standardize layer.
Leser JSON-filer fra Landing layer og konverterer til flate Delta-tabeller i Standardize.
Leser JSON-filer fra Landing layer og konverterer til flate Delta-tabeller i Standardize layer.

## Hva notebooken gjør
- Leser `pipeline.ssb_load_queue` for å vite hvilke tabeller som skal prosesseres
- For hver tabell: finn nyeste snapshot i Landing, les alle periode-filer
- Parser JSON-Stat2 til flat Spark DataFrame med `_code`/`_label`-kolonner
- Beriker med kommunekorrespondanse (2024-nummere)
- Kjører datakvalitetssjekker
- Skriver til Delta-tabell partisjonert på `year`
- Kjører `nb_refine_{table_id}` hvis den finnes (tabellspesifikk raffinering)

## Kjøring
Kjøres fra Fabric Pipeline etter `04_ssb_ingest_landing`.
Parameteren `QUEUE_TABLE` settes av pipeline.

---------


In [ ]:
# =====================================================================
# PARAMETERE (overstyres av Fabric Pipeline)
QUEUE_TABLE          = "pipeline.ssb_load_queue"   # eller pipeline.pipeline.ssb_load_queue_critical
SNAPSHOT_DATE        = None               # None = bruk nyeste snapshot
WRITE_MODE           = "overwrite"        # overwrite eller append

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [CapacityLimitExceeded] Unable to complete the action because your organization’s Fabric compute capacity has exceeded its limits. Try again later. HTTP status code: 429.

In [ ]:
# =====================================================================
# IMPORTS
from __future__ import annotations

import itertools
import json
import re
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType

spark = SparkSession.builder.appName("SSBStandardize").getOrCreate()

try:
    from notebookutils import mssparkutils
    FABRIC_AVAILABLE = True
except ImportError:
    mssparkutils = None
    FABRIC_AVAILABLE = False
    print("mssparkutils ikke tilgjengelig - kjorer i test-modus")

print(f"Imports OK | Fabric: {FABRIC_AVAILABLE}")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# KONFIGURASJON
LAKEHOUSE_ROOT = "Files"
ZONE_LANDING   = "landing"
DATA_SOURCE    = "ssb"
DATA_PRODUCT   = "statbank"

# Dimensjonstabell for kommunekorrespondanser
DIM_KOMMUNE_PATH = "abfss://AFKA-DataEng@onelake.dfs.fabric.microsoft.com/lh_shared_resources.lakehouse/Tables/dim_kommunekorrespondanser"

print("Konfigurasjon OK")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# FABRIC FILESYSTEM HELPERS

class FabricFS:
    """Filoperasjoner – fungerer både i Fabric og lokalt."""

    @staticmethod
    def normalize_path(path: str) -> str:
        if path.startswith("Files/"):
            return path
        if path.startswith("/lakehouse/default/Files/"):
            return path.replace("/lakehouse/default/", "")
        return path

    @staticmethod
    def exists(path: str) -> bool:
        path = FabricFS.normalize_path(path)
        if FABRIC_AVAILABLE:
            try:
                mssparkutils.fs.ls(path)
                return True
            except Exception:
                return False
        else:
            import os
            return os.path.exists(path)

    @staticmethod
    def read_json(path: str) -> dict:
        """
        Les JSON – bruker Spark for store filer, fallback til mssparkutils eller lokal fil.
        """
        path = FabricFS.normalize_path(path)
        if FABRIC_AVAILABLE:
            try:
                # Use Spark if file is large enough for distributed reading
                try:
                    rdd = spark.sparkContext.textFile(path)
                    return json.loads("\n".join(rdd.collect()))
                except Exception:
                    # For small files or if Spark fails, fallback to mssparkutils
                    content = mssparkutils.fs.head(path, 500_000_000)
                    return json.loads(content)
            except Exception:
                pass
        # Local fallback
        import os
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        raise FileNotFoundError(f"File not found: {path}")

    @staticmethod
    def list_subdirs(path: str) -> list:
        """List undermapper, returner mappenavn (ikke full sti)."""
        path = FabricFS.normalize_path(path)
        if FABRIC_AVAILABLE:
            try:
                return [item.name for item in mssparkutils.fs.ls(path) if item.isDir]
            except Exception:
                return []
        else:
            import os
            if not os.path.exists(path):
                return []
            return [d for d in os.listdir(path)
                    if os.path.isdir(os.path.join(path, d))]


def get_latest_snapshot_path(table_id: str) -> Optional[str]:
    """
    Finn nyeste snapshot-mappe for en tabell.
    Returnerer full relativ sti, eller None hvis ingen snapshot finnes.
    """
    landing_base = f"{LAKEHOUSE_ROOT}/{ZONE_LANDING}/{DATA_SOURCE}/{DATA_PRODUCT}/{table_id}"
    subdirs = FabricFS.list_subdirs(landing_base)
    snapshots = sorted(
        [d for d in subdirs if d.startswith("snapshot_date=")],
        reverse=True,
    )
    if not snapshots:
        return None
    target = SNAPSHOT_DATE or snapshots[0].replace("snapshot_date=", "")
    for snap in snapshots:
        if snap == f"snapshot_date={target}":
            return f"{landing_base}/{snap}"
    return None

def get_period_files(snapshot_path: str, table_id: str) -> list:
    """
    Finn alle periode-filer i et snapshot.
    Returnerer liste av (period, file_path).
    Avhenger ikke av manifest – leser mappestrukturen direkte.
    """
    period_dirs = FabricFS.list_subdirs(snapshot_path)
    results = []
    for d in sorted(period_dirs):
        if not d.startswith("period="):
            continue
        period = d.replace("period=", "")
        file_path = f"{snapshot_path}/{d}/ssb_{table_id}_{period}.json"
        if FabricFS.exists(file_path):
            results.append((period, file_path))
    return results

print("FabricFS OK")


StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# TIDSKODE PARSING

def parse_period_to_parts(period: str) -> Dict:
    """
    Parser tidskode til year, quarter, month, week, period_type.
    Brukes for partisjonering og analyse.
    """
    period = str(period).strip()

    m = re.match(r"(\d{4})[KkQq](\d{1})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": int(m.group(2)),
                "month": None, "week": None, "period_type": "quarter"}

    m = re.match(r"(\d{4})[Mm](\d{2})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": None,
                "month": int(m.group(2)), "week": None, "period_type": "month"}

    m = re.match(r"(\d{4})[UuWw](\d{2})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": None,
                "month": None, "week": int(m.group(2)), "period_type": "week"}

    m = re.match(r"(\d{4})[Hh](\d{1})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": None, "month": None,
                "week": None, "period_type": "half_year"}

    if "-" in period:
        year_str = period.split("-")[0]
    else:
        year_str = period

    m = re.search(r"(\d{4})", year_str)
    return {"year": int(m.group(1)) if m else None,
            "quarter": None, "month": None, "week": None, "period_type": "year"}

print("Tidskode-parsing OK")


StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# GEO-NIVÅ KLASSIFISERING

def classify_geo_level(region_code: str) -> Optional[str]:
    """
    Klassifiser geografisk niva basert pa region_code-lengde.
    1 siffer  -> N (Nasjonalt)
    2 siffer  -> F (Fylke)
    4 siffer  -> K (Kommune)
    6 siffer  -> B (Bydel)
    """
    if not region_code:
        return None
    code = str(region_code).strip()
    if not code.isdigit():
        if code and code[-1].isalpha():
            code = code[:-1]
        else:
            return None
    return {1: "N", 2: "F", 4: "K", 6: "B"}.get(len(code))

print("Geo-niva OK")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# JSON-STAT2 PARSER

def parse_jsonstat2_to_dataframe(
    json_data: dict,
    period: str,
    snapshot_date: str,
    table_id: str,
) -> DataFrame:
    """
    Parser JSON-Stat2 til flat Spark DataFrame.

    Hver dimensjon far to kolonner: {dim}_code og {dim}_label.
    Tidsdimensjonen far ekstra kolonner: year, quarter, month, week, period_type.
    Geografisk dimensjon far geo_level (N/F/K/B).
    """
    dimensions = json_data.get("dimension", {})
    dim_ids    = list(dimensions.keys())
    values     = json_data.get("value", [])

    sizes = [
        len(dimensions[d]["category"]["index"])
        for d in dim_ids
    ]

    records = []
    for flat_idx, indices in enumerate(itertools.product(*[range(s) for s in sizes])):
        if flat_idx >= len(values):
            break

        raw_val = values[flat_idx]
        record = {
            "table_id":     table_id,
            "snapshot_date": snapshot_date,
            "period":       period,
            "value":        float(raw_val) if raw_val is not None else None,
        }

        region_code = None
        for dim_idx, dim_id in enumerate(dim_ids):
            cat_idx  = indices[dim_idx]
            dim_data = dimensions[dim_id]
            cats     = list(dim_data["category"]["index"].keys())
            labels   = dim_data["category"]["label"]
            code     = cats[cat_idx]
            label    = labels.get(code, code)
            col      = dim_id.lower().replace(" ", "_")
            record[f"{col}_code"]  = code
            record[f"{col}_label"] = label
            if col == "region":
                region_code = code

        record["geo_level"] = classify_geo_level(region_code) if region_code else None
        records.append(record)

    df = spark.createDataFrame(records)

    # Legg til tidsdimensjoner
    tp = parse_period_to_parts(period)
    df = (
        df
        .withColumn("year",        F.lit(tp["year"]).cast(IntegerType()))
        .withColumn("quarter",     F.lit(tp["quarter"]).cast(IntegerType()))
        .withColumn("month",       F.lit(tp["month"]).cast(IntegerType()))
        .withColumn("week",        F.lit(tp["week"]).cast(IntegerType()))
        .withColumn("period_type", F.lit(tp["period_type"]))
        .withColumn("processed_at", F.current_timestamp())
    )
    return df

print("JSON-Stat2 parser OK")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# KOMMUNE 2024-BERIKELSE

def enrich_with_kommune_2024(df: DataFrame) -> DataFrame:
    """
    Berikar med oppdatert kommunenummer (2024) fra dimensjonstabellen.
    Left join pa region_code -> kommunenr_historiske.
    Tabeller uten region-dimensjon returneres uendret med region_code_24 = None.
    """
    if "region_code" not in df.columns:
        print("   Ingen region_code-kolonne – hopper over kommune-berikelse")
        return df.withColumn("region_code_24", F.lit(None).cast(StringType()))

    try:
        df_dim = (
            spark.read.format("delta").load(DIM_KOMMUNE_PATH)
            .select("kommunenr_historiske", "kommunenr_2024")
            .distinct()
        )
        df_enriched = (
            df
            .join(df_dim, df["region_code"] == df_dim["kommunenr_historiske"], "left")
            .drop("kommunenr_historiske")
            .withColumnRenamed("kommunenr_2024", "region_code_24")
        )
        mapped = df_enriched.filter(F.col("region_code_24").isNotNull()).count()
        total  = df.count()
        print(f"   Kommune-berikelse: {mapped:,}/{total:,} rader mappet ({mapped/total:.1%})")
        return df_enriched
    except Exception as e:
        print(f"   ADVARSEL: Kommune-berikelse feilet ({e}) – fortsetter uten")
        return df.withColumn("region_code_24", F.lit(None).cast(StringType()))

print("Kommune-berikelse OK")


StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# DATAKVALITETSSJEKKER

def run_data_quality_checks(df: DataFrame) -> Dict:
    """Grunnleggende datakvalitetssjekker."""
    total     = df.count()
    nulls     = df.filter(F.col("value").isNull()).count()
    null_rate = nulls / total if total > 0 else 0

    key_cols   = [c for c in df.columns if c.endswith("_code")] + ["period"]
    duplicates = (
        df.groupBy(key_cols).count().filter(F.col("count") > 1).count()
        if key_cols else 0
    )

    year_min = df.agg(F.min("year")).collect()[0][0]
    year_max = df.agg(F.max("year")).collect()[0][0]

    checks = {
        "total_rows":    total,
        "null_values":   nulls,
        "null_rate":     round(null_rate, 4),
        "duplicates":    duplicates,
        "year_min":      year_min,
        "year_max":      year_max,
        "has_issues":    null_rate > 0.15 or duplicates > 0,
    }

    print(f"   Rader: {total:,}  |  Null: {nulls:,} ({null_rate:.1%})  |  "
          f"Duplikater: {duplicates}  |  Ar: {year_min}-{year_max}")
    if checks["has_issues"]:
        print("   ADVARSEL: Datakvalitetsproblemer funnet")

    return checks

print("Datakvalitet OK")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# SKRIV TIL SILVER

def write_to_delta(df: DataFrame, target_table: str) -> None:
    """
    Skriv til Delta-tabell i Standardize layer, partisjonert pa year.
    """
    # Sorter kolonner for lesbarhet
    meta_cols = ["table_id", "snapshot_date", "period",
                 "year", "quarter", "month", "week", "period_type"]
    dim_cols  = sorted([c for c in df.columns if c.endswith(("_code", "_label"))])
    val_cols  = [c for c in ["value", "geo_level", "region_code_24", "processed_at"]
                 if c in df.columns]
    ordered   = [c for c in meta_cols + dim_cols + val_cols if c in df.columns]
    df        = df.select(ordered)

    (
        df.write
        .format("delta")
        .mode(WRITE_MODE)
        .partitionBy("year")
        .option("overwriteSchema", "true")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )
    print(f"   Skrevet til {target_table}")

    # OPTIMIZE for bedre leseytelse
    try:
        spark.sql(f"OPTIMIZE {target_table}")
        print(f"   OPTIMIZE fullfort")
    except Exception as e:
        print(f"   ADVARSEL: OPTIMIZE feilet (ikke kritisk): {e}")

print("Skriving OK")


StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# PROSESSER ÉN TABELL

def process_table(table_id: str) -> dict:
    """
    Les alle perioder fra nyeste snapshot og skriv til Standardize.
    """
    target_table = f"ssb.statbank_{table_id}"
    print(f"\n{'='*60}")
    print(f"Tabell: {table_id}  ->  {target_table}")
    print(f"{'='*60}")

    result = {
        "status":       "unknown",
        "table_id":     table_id,
        "target_table": target_table,
        "rows":         0,
        "periods":      0,
        "error":        None,
    }
    try:
        # Finn snapshot
        snapshot_path = get_latest_snapshot_path(table_id)
        if not snapshot_path:
            raise ValueError(f"Ingen snapshot funnet for {table_id} i Landing layer")
        snapshot_date = snapshot_path.split("snapshot_date=")[-1]
        print(f"Snapshot: {snapshot_date}")
        # Finn periode-filer direkte fra mappestruktur (uavhengig av manifest)
        period_files = get_period_files(snapshot_path, table_id)
        if not period_files:
            raise ValueError(f"Ingen periode-filer funnet i {snapshot_path}")
        print(f"Perioder: {len(period_files)} ({period_files[0][0]} -> {period_files[-1][0]})")
        # Les og parser alle perioder
        dfs = []
        for idx, (period, file_path) in enumerate(period_files, 1):
            try:
                json_data = FabricFS.read_json(file_path)
                df_period = parse_jsonstat2_to_dataframe(
                    json_data, period, snapshot_date, table_id
                )
                dfs.append(df_period)
                if idx % 10 == 0 or idx == len(period_files):
                    print(f"   [{idx}/{len(period_files)}] {period} OK")
            except Exception as e:
                print(f"   [{idx}/{len(period_files)}] {period} FEIL: {e}")
                continue
        if not dfs:
            raise ValueError("Ingen perioder kunne prosesseres")
        # Union alle perioder
        df_final = dfs[0]
        for df in dfs[1:]:
            df_final = df_final.unionByName(df, allowMissingColumns=True)
        # Berikning og kvalitetssjekk
        df_final = enrich_with_kommune_2024(df_final)
        print("Datakvalitet:")
        quality  = run_data_quality_checks(df_final)
        # Skriv til Standardize
        write_to_delta(df_final, target_table)
        # Kjor tabellspesifikk refine-notebook hvis den finnes
        if FABRIC_AVAILABLE:
            refine_nb = f"nb_refine_{table_id}"
            try:
                mssparkutils.notebook.run(refine_nb, 1800, {"TABELLNR": table_id})
                print(f"   {refine_nb} fullfort")
            except Exception as e:
                err = str(e)
                if "Fetch notebook content" in err or "NotebookExecutionException" in err:
                    print(f"   {refine_nb} finnes ikke – hopper over")
                else:
                    print(f"   ADVARSEL: {refine_nb} feilet: {err}")
        result["status"]  = "success"
        result["rows"]    = quality["total_rows"]
        result["periods"] = len(dfs)
        return result
    except Exception as e:
        import traceback
        print(f"\nFEIL: {e}")
        traceback.print_exc()
        result["status"] = "failed"
        result["error"]  = str(e)
    return result

print("process_table OK")


StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# =====================================================================
# HOVEDLOOP – prosesser alle tabeller fra koen

start_time = time.time()

print(f"Ko: {QUEUE_TABLE}")
print(f"Write mode: {WRITE_MODE}")
print("="*60)

if not spark.catalog.tableExists(QUEUE_TABLE):
    raise RuntimeError(
        f"Ko-tabell '{QUEUE_TABLE}' finnes ikke. "
        "Kjor 03_ssb_oppdateringsdetector og 04_ssb_ingest_landing forst."
    )

# Hent unike tabeller fra koen (deduper pa table_id)
queue_rows = (
    spark.table(QUEUE_TABLE)
    .select("table_id")
    .distinct()
    .collect()
)

if not queue_rows:
    print("Ko er tom – ingen tabeller a prosessere. Avslutter.")
else:
    print(f"Fant {len(queue_rows)} tabeller i koen\n")

    results    = []
    ok_count   = 0
    fail_count = 0

    for row in queue_rows:
        result = process_table(row["table_id"])
        results.append(result)
        if result["status"] == "success":
            ok_count += 1
        else:
            fail_count += 1

    elapsed = int(time.time() - start_time)

    print(f"\n{'='*60}")
    print(f"SAMMENDRAG  ({elapsed}s totalt)")
    print(f"{'='*60}")
    print(f"  Vellykket: {ok_count}")
    print(f"  Feilet:    {fail_count}")
    print()

    for r in results:
        icon = "OK" if r["status"] == "success" else "!!"
        print(
            f"  [{icon}] {r['table_id']}: "
            f"{r['target_table']:<35} "
            f"{r['rows']:>10,} rader, {r['periods']} perioder"
        )

    if fail_count > 0:
        print("\nFeil-detaljer:")
        for r in results:
            if r["status"] == "failed":
                print(f"  {r['table_id']}: {r['error']}")

    print("SILVER TRANSFORMASJON FULLFORT")
    print(f"{'='*60}")

StatementMeta(, , -1, Cancelled, , Cancelled, True)